In [1]:
# --- Step 1: نصب پکیج‌های لازم (Colab)
!pip install -q transformers sentence-transformers faiss-cpu gradio datasets

# چک اولیه در پایتون
import sys
import torch
print("Python:", sys.version.splitlines()[0])
print("torch:", getattr(torch, "__version__", "n/a"), "| CUDA available:", torch.cuda.is_available())

# تست ایمپورت‌ها و نسخه‌ها (اطلاعات مفید برای دیباگ)
try:
    import transformers
    print("transformers:", transformers.__version__)
except Exception as e:
    print("transformers import error:", e)

try:
    import sentence_transformers
    print("sentence-transformers:", sentence_transformers.__version__)
except Exception as e:
    print("sentence-transformers import error:", e)

# faiss گاهی نصب میشه ولی import خطا بده؛ اینجا چک می‌کنیم
try:
    import faiss
    print("faiss:", getattr(faiss, "__version__", "version attribute missing"))
except Exception as e:
    print("faiss import error (not fatal):", e)

try:
    import gradio as gr
    print("gradio:", gr.__version__)
except Exception as e:
    print("gradio import error:", e)

try:
    import datasets
    print("datasets (huggingface):", datasets.__version__)
except Exception as e:
    print("datasets import error:", e)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 32.9 MB/s eta 0:00:00
Python: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
torch: 2.8.0+cu126 | CUDA available: True
transformers: 4.56.1
sentence-transformers: 5.1.0
faiss: 1.12.0
gradio: 5.46.0
datasets (huggingface): 4.0.0


In [2]:
from datasets import load_dataset

# مرحله ۲: بارگذاری دیتاست نمونه (می‌تونیم بعداً عوضش کنیم)
dataset = load_dataset("ag_news")

# نمایش چند نمونه برای درک ساختار
print(dataset)
print("\nیک نمونه:")
print(dataset["train"][0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

یک نمونه:
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


In [3]:
from sentence_transformers import SentenceTransformer

# مرحله ۳: بارگذاری مدل امبدینگ
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# برای تست: یک جمله را امبد کنیم
sample_text = dataset["train"][0]["text"]
embedding = embedder.encode(sample_text)

print("متن نمونه:", sample_text)
print("شکل بردار امبدینگ:", embedding.shape)  # باید یک آرایه ۳۸۴ بُعدی باشه
print("چندتا عدد اول:", embedding[:10])


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

متن نمونه: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
شکل بردار امبدینگ: (384,)
چندتا عدد اول: [ 0.00743859  0.02856239  0.04109549  0.1050014   0.02328203  0.03512586
 -0.02140599 -0.02290231  0.00494522 -0.06689856]


In [4]:
import faiss
import numpy as np

# چند نمونه برای تست (مثلاً 2000 تا)
texts = dataset["train"]["text"][:2000]

# تبدیل به امبدینگ
embeddings = embedder.encode(texts, show_progress_bar=True)

# تبدیل به آرایه NumPy با نوع float32 (لازم برای FAISS)
embeddings = np.array(embeddings, dtype="float32")

# ابعاد بردار (384 باید باشه)
d = embeddings.shape[1]

# ساخت ایندکس FAISS (L2 distance)
index = faiss.IndexFlatL2(d)
index.add(embeddings)

print("تعداد بردارها در ایندکس:", index.ntotal)


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

تعداد بردارها در ایندکس: 2000


In [5]:
from sentence_transformers import SentenceTransformer

# بارگذاری مدل امبدینگ
embedder = SentenceTransformer("all-MiniLM-L6-v2")


In [6]:
# تابع برای جستجو
def search(query, top_k=3):
    # امبدینگ پرسش
    query_vec = embedder.encode([query]).astype("float32")

    # جستجو در ایندکس
    distances, indices = index.search(query_vec, top_k)

    # نمایش نتایج
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "rank": i+1,
            "text": texts[idx],
            "distance": float(distances[0][i])
        })
    return results

# تست با یک پرسش
query = "Space technology and NASA missions"
results = search(query, top_k=3)

for r in results:
    print(f"Rank {r['rank']} | Distance: {r['distance']:.4f}")
    print(r['text'])
    print("-"*80)


Rank 1 | Distance: 0.8540
Redesigning Rockets: NASA Space Propulsion Finds a New Home (SPACE.com) SPACE.com - While the exploration of the Moon and other planets in our solar system is nbsp;exciting, the first task for astronauts and robots alike is to actually nbsp;get to those destinations.
--------------------------------------------------------------------------------
Rank 2 | Distance: 0.8914
New NASA Supercomputer to Aid Theorists and Shuttle Engineers (SPACE.com) SPACE.com - NASA researchers have teamed up with a pair of Silicon Valley firms to build \  a supercomputer that ranks alongside the world's largest Linux-based systems.
--------------------------------------------------------------------------------
Rank 3 | Distance: 1.0045
Space Science Pioneer Van Allen Questions Human Spaceflight (SPACE.com) SPACE.com - A leading space scientist has called to question the validity of human spaceflight, suggesting that sending astronauts outward from Earth is outdated, too costly, a

In [7]:
def search(query, top_k=3):
    # امبدینگ پرسش
    query_vec = embedder.encode([query]).astype("float32")

    # جستجو در ایندکس
    distances, indices = index.search(query_vec, top_k)

    # نمایش نتایج
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "rank": i+1,
            "text": texts[idx],
            "distance": float(distances[0][i])
        })
    return results


In [8]:
import gradio as gr

def chatbot(query):
    results = search(query, top_k=3)
    answer = ""
    for r in results:
        answer += f"🔹 Rank {r['rank']} | Distance: {r['distance']:.4f}\n"
        answer += r['text'] + "\n\n"
    return answer

# رابط کاربری
demo = gr.Interface(
    fn=chatbot,
    inputs=gr.Textbox(lines=2, placeholder="پرسش خود را اینجا بنویسید..."),
    outputs="text",
    title="📚 Chatbot Demo with Semantic Search",
    description="یک چت‌بات ساده با امبدینگ + FAISS"
)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://10804a27708b6568e4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [10]:
import gradio as gr

def chat_ui(history, message):
    response = chatbot(message)   # همون تابعی که نوشتیم
    history.append((message, response))
    return history, ""

with gr.Blocks(css=".gradio-container {width: 900px !important;}") as demo:
    gr.Markdown("## 🤖 Chatbot with FAISS Search")
    chatbot_ui = gr.Chatbot()
    msg = gr.Textbox(placeholder="سوالتو بپرس...", lines=2)
    clear = gr.Button("پاک کردن گفتگو")

    msg.submit(chat_ui, [chatbot_ui, msg], [chatbot_ui, msg])
    clear.click(lambda: [], None, chatbot_ui)

demo.launch()


/tmp/ipython-input-3664721198.py:10: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot_ui = gr.Chatbot()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://19d5f9b0a25f3942de.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [9]:
# تست مستقیم بدون Gradio
query = "NASA and space mission"
print(search(query, top_k=3))


[{'rank': 1, 'text': 'Redesigning Rockets: NASA Space Propulsion Finds a New Home (SPACE.com) SPACE.com - While the exploration of the Moon and other planets in our solar system is nbsp;exciting, the first task for astronauts and robots alike is to actually nbsp;get to those destinations.', 'distance': 0.8540733456611633}, {'rank': 2, 'text': 'Space Science Pioneer Van Allen Questions Human Spaceflight (SPACE.com) SPACE.com - A leading space scientist has called to question the validity of human spaceflight, suggesting that sending astronauts outward from Earth is outdated, too costly, and the science returned is trivial.', 'distance': 0.9664839506149292}, {'rank': 3, 'text': "New NASA Supercomputer to Aid Theorists and Shuttle Engineers (SPACE.com) SPACE.com - NASA researchers have teamed up with a pair of Silicon Valley firms to build \\  a supercomputer that ranks alongside the world's largest Linux-based systems.", 'distance': 1.0112838745117188}]
